# Yield Curve Terminal — code walkthrough

A **runnable tour of the analytics engine** behind the app. Run the cells top to bottom.

> The interactive app itself runs with `streamlit run app.py` — it **cannot** run inside Jupyter (it's a Streamlit app). This notebook imports the *same* `analytics.py` the app uses and exercises each piece so you can see what it does. Launch Jupyter from inside this folder so the import works.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())
import numpy as np, pandas as pd, datetime as dt
import analytics as A

## 1. Fetch the live curve (FRED)
`build_curve` downloads every Treasury maturity from FRED and returns **today's curve** plus the **full daily history**.

In [ ]:
today = dt.date.today()
start = (today - dt.timedelta(days=365*8)).strftime('%Y-%m-%d')
end   = today.strftime('%Y-%m-%d')
latest, wide = A.build_curve('US', start, end)
latest

## 2. Interpolation & spreads
`make_interpolator` turns the ~11 quoted points into a continuous function, so you can price any maturity (e.g. 4.5Y) and any spread.

In [ ]:
interp, xs, ys = A.make_interpolator(latest, 'linear')
print('4.5Y yield =', round(float(interp(4.5)), 3), '%')
A.custom_spreads(interp, [(2, 10), (5, 30)])

## 3. Nelson-Siegel-Svensson fit
Fits the whole curve with a few interpretable parameters and reads **level / slope / curvature** off the smooth fit.

In [ ]:
nss = A.fit_nss(latest)
print('level=%.2f  short=%.2f  slope=%+.2f  curvature=%+.2f  RMSE=%.1f bps'
      % (nss['level'], nss['short_rate'], nss['slope'], nss['curvature'], nss['rmse_bps']))
print('NSS-implied 10Y =', round(nss['eval'](10), 3), '%')

## 4. Forward rates & the market-implied path
The curve implies future rates by no-arbitrage — the market's own forecast.

In [ ]:
t, spot, fwd = A.forward_curve(interp, 1.0, xs.min(), 30.0)
starts, implied, spot0 = A.implied_short_rate_path(interp, max_year=10, step=1.0)
print('today 1Y =', round(spot0, 2), '%')
print('implied 1Y rate, each year forward:', np.round(implied, 2))

## 5. PCA — the level / slope / curvature factors
PCA on daily curve *changes*: three shapes explain almost every move.

In [ ]:
pca = A.pca_factors(wide, 3)
for name, ev in zip(pca['names'], pca['explained']):
    print(f'{name:<10} {ev:.0%} of daily curve moves')

## 6. Spread mean reversion
z-score vs history and AR(1) half-life for a spread.

In [ ]:
sp = A.spread_series(wide, 2, 10)
mr = A.mean_reversion(sp)
print('2s10s now=%+.0f bps  mean=%+.0f  z=%+.2f  half-life=%.0f trading days'
      % (mr['current'], mr['mean'], mr['z'], mr['half_life_days']))

## 7. Breakeven inflation (US TIPS)
Nominal minus real (TIPS) = the inflation the market prices.

In [ ]:
tips_latest, tips_wide = A.build_tips_curve('US', start, end)
be = A.breakevens(interp, tips_latest)
be

## 8. Scenario shocks
The classic curve deformations (yield deltas per maturity, in %).

In [ ]:
shocks = A.shock_profiles(latest['Maturity (Years)'].values, 0.5)  # 50 bp magnitude
list(shocks.keys())

## 9. Bond risk — duration, DV01, convexity, key-rate, scenario P&L
The **Rates Risk** tab: price a bond off the curve and measure its rate risk (all by bump-and-reprice).

In [ ]:
risk = A.bond_risk(latest, coupon=4.0, maturity=10, notional=1_000_000)
print('price=%.2f per100   Mod.Duration=%.2f y   DV01=$%.0f   Convexity=%.1f'
      % (risk['per100'], risk['mod_duration'], risk['dv01'], risk['convexity']))

In [ ]:
krd = A.key_rate_dv01(latest, 4.0, 10, 1_000_000)
print('Key-rate DV01 (sums to total DV01):')
for m, v in krd:
    print(f'  {m:>4}Y : ${v:,.0f} per 1bp')

In [ ]:
_, rows = A.scenario_bond_pnl(latest, shocks, 4.0, 10, 1_000_000, risk=risk)
for r in rows:
    print(f"  {r['scenario']:<14} P&L = ${r['pnl']:+,.0f}")

## Where the rest of the code lives
| File | What it holds |
|---|---|
| `analytics.py` | everything above (pure functions) |
| `charts.py` | the Plotly charts |
| `app.py` | the Streamlit UI / layout |
| `content.py` | the in-app Guide & 'how to read' text |
| `theme.py` | colours / styling |

**Run the full interactive app:** `streamlit run app.py` in a terminal.